In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
!pip install pandas numpy openpyxl sentence-transformers transformers torch spacy faiss-cpu anthropic flask flask-cors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 49.1 MB/s eta 0:00:00


In [7]:
import pandas as pd
import numpy as np
import spacy
import torch

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import faiss

In [9]:
import pandas as pd
import json

# Load raw data
df = pd.read_excel("/content/edufeed_clean_data.xlsx")

# Basic cleanup
df = df.dropna()

# Save ingested CSV
df.to_csv("layer1_ingested.csv", index=False)

# Create report
report = {
    "rows": len(df),
    "columns": list(df.columns),
    "missing_values": df.isnull().sum().to_dict()
}

# Save report JSON
with open("layer1_report.json", "w") as f:
    json.dump(report, f, indent=4)

print("Layer 1 done")

Layer 1 done


In [ ]:
import pandas as pd
import json

df = pd.read_csv("layer1_ingested.csv")

# Example anonymization (adjust columns if needed)
mapping = {}

if "name" in df.columns:
    mapping["name"] = df["name"].tolist()
    df["name"] = "user_" + df.index.astype(str)

if "email" in df.columns:
    mapping["email"] = df["email"].tolist()
    df["email"] = "hidden"

# Save anonymized CSV
df.to_csv("layer2_anonymized.csv", index=False)

# Save mapping JSON
with open("layer2_map.json", "w") as f:
    json.dump(mapping, f, indent=4)

print("Layer 2 done")

In [10]:
import pandas as pd
import json

df = pd.read_csv("layer1_ingested.csv")

mapping = {
    "name_map": {},
    "email_map": {}
}

# Example anonymization (adjust columns if needed)

if "name" in df.columns:
    for i, val in enumerate(df["name"].astype(str)):
        mapping["name_map"][val] = f"user_{i}"
    df["name"] = df["name"].astype(str).apply(lambda x: mapping["name_map"].get(x, x))

if "email" in df.columns:
    for val in df["email"].astype(str):
        mapping["email_map"][val] = "hidden"
    df["email"] = "hidden"

# Save anonymized dataset
df.to_csv("layer2_anonymized.csv", index=False)

# Save PROF MAP JSON (your required file)
with open("layer2_prof_map.json", "w") as f:
    json.dump(mapping, f, indent=4)

print("Layer 2 completed")

Layer 2 completed


In [11]:

"""
Layer 1 — Data Ingestion (FIXED FOR COLAB)
"""

import logging
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

# ── COLAB SAFE PATH ──────────────────────────────────────────────────────────
ROOT = Path("/content")

RAW_PATH    = ROOT / "edufeed_clean_data.xlsx"
OUT_CSV     = ROOT / "layer1_ingested.csv"
REPORT_PATH = ROOT / "layer1_report.json"
LOG_PATH    = ROOT / "layer1.log"

# ── logging ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_PATH),
        logging.StreamHandler(),
    ],
)

log = logging.getLogger(__name__)

# ── schema ────────────────────────────────────────────────────────────────────
REQUIRED_COLUMNS = [
    "professor_name", "school_name", "department_name",
    "star_rating", "comments", "post_date",
    "student_star", "student_difficult",
    "sentiment_label", "tag_count", "course_mode",
]

COLUMN_TYPES = {
    "star_rating": float,
    "student_star": float,
    "student_difficult": float,
    "tag_count": int,
}

RATING_BOUNDS = {
    "star_rating": (1.0, 5.0),
    "student_star": (1.0, 5.0)
}

# ── helpers ───────────────────────────────────────────────────────────────────

def load_raw(path: Path) -> pd.DataFrame:
    if path.suffix == ".xlsx":
        df = pd.read_excel(path, engine="openpyxl")
    else:
        df = pd.read_csv(path)

    log.info(f"Loaded {len(df)} rows")
    return df


def validate_schema(df):
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        log.warning(f"Missing columns: {missing}")
    return missing


def coerce_types(df):
    for col, dtype in COLUMN_TYPES.items():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            if dtype == int:
                df[col] = df[col].astype("Int64")
    return df


def drop_nulls(df):
    before = len(df)
    df = df.dropna(subset=["professor_name", "comments", "star_rating"])
    log.info(f"Dropped {before - len(df)} null rows")
    return df, before - len(df)


def deduplicate(df):
    before = len(df)
    df = df.drop_duplicates(subset=["professor_name", "comments", "post_date"])
    log.info(f"Removed {before - len(df)} duplicates")
    return df, before - len(df)


def validate_bounds(df):
    violations = {}
    for col, (lo, hi) in RATING_BOUNDS.items():
        if col in df.columns:
            bad = df[(df[col] < lo) | (df[col] > hi)]
            if len(bad):
                violations[col] = len(bad)
    return violations


def add_row_id(df):
    df = df.reset_index(drop=True)
    df.insert(0, "review_id", df.index.astype(str).str.zfill(6))
    return df

# ── main pipeline ─────────────────────────────────────────────────────────────

def run():
    log.info("Layer 1 starting...")

    df = load_raw(RAW_PATH)
    raw_rows = len(df)

    missing_cols = validate_schema(df)
    df = coerce_types(df)

    df, null_removed = drop_nulls(df)
    df, dedup_removed = deduplicate(df)

    violations = validate_bounds(df)

    df = add_row_id(df)

    df.to_csv(OUT_CSV, index=False)

    report = {
        "run_at": str(datetime.utcnow()),
        "raw_rows": raw_rows,
        "clean_rows": len(df),
        "null_removed": null_removed,
        "dedup_removed": dedup_removed,
        "missing_columns": missing_cols,
        "violations": violations
    }

    with open(REPORT_PATH, "w") as f:
        json.dump(report, f, indent=2)

    log.info("Layer 1 completed")

    return df


# ── RUN ──────────────────────────────────────────────────────────────────────
df_final = run()
df_final.head()

/tmp/ipykernel_835/2991598114.py:131: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "run_at": str(datetime.utcnow()),


,review_id,professor_name,school_name,department_name,local_name,state_name,year_since_first_review,star_rating,take_again,diff_index,...,post_month,date_missing_flag,grade_group,sentiment_label,help_ratio,difficulty_category,experience_bucket,comment_length_cat,tag_count,course_mode
0,000000,Leslie Looney,University Of Illinois At Urbana-Champaign,Astronomy Department,Champaign\Xe2\X80\X93Urbana,Il,11,4.7,0.75,2.0,...,6.0,0,B Range,Positive,0.0,Moderate,10+ yrs,Medium,3,Offline
1,000001,Leslie Looney,University Of Illinois At Urbana-Champaign,Astronomy Department,Champaign\Xe2\X80\X93Urbana,Il,11,4.7,0.75,2.0,...,4.0,0,A Range,Positive,0.0,Easy,10+ yrs,Medium,3,Offline
2,000002,Leslie Looney,University Of Illinois At Urbana-Champaign,Astronomy Department,Champaign\Xe2\X80\X93Urbana,Il,11,4.7,0.75,2.0,...,7.0,0,Not Reported,Positive,0.0,Moderate,10+ yrs,Medium,3,Offline
3,000003,Leslie Looney,University Of Illinois At Urbana-Champaign,Astronomy Department,Champaign\Xe2\X80\X93Urbana,Il,11,4.7,0.75,2.0,...,8.0,0,A Range,Positive,0.0,Moderate,10+ yrs,Long,3,Offline
4,000004,Leslie Looney,University Of Illinois At Urbana-Champaign,Astronomy Department,Champaign\Xe2\X80\X93Urbana,Il,11,4.7,0.75,2.0,...,2.0,0,Not Reported,Positive,0.0,Easy,10+ yrs,Long,3,Offline


In [15]:
import re
import hashlib
import json
from pathlib import Path
import pandas as pd

ROOT = Path("/content")

IN_CSV = ROOT / "layer1_ingested.csv"
OUT_CSV = ROOT / "layer2_anonymised.csv"
MAP_PATH = ROOT / "layer2_prof_map.json"

def sha256_token(v):
    return "PROF_" + hashlib.sha256(v.lower().strip().encode()).hexdigest()[:8].upper()

def build_map(df):
    return {n: sha256_token(n) for n in df["professor_name"].dropna().unique()}

def run():
    df = pd.read_csv(IN_CSV)

    prof_map = build_map(df)

    with open(MAP_PATH, "w") as f:
        json.dump(prof_map, f, indent=2)

    df["professor_token"] = df["professor_name"].map(prof_map)
    df = df.drop(columns=["professor_name"])

    comments = df["comments"].fillna("").astype(str)

    patterns = []
    for name in prof_map.keys():
        for p in name.split():
            if len(p) > 2:
                patterns.append(p)

    if patterns:
        regex = re.compile(r"\b(" + "|".join(map(re.escape, patterns)) + r")\b", re.I)
        df["comments"] = comments.str.replace(regex, "[PROF]", regex=True)
    else:
        df["comments"] = comments

    for col in ["school_name", "local_name"]:
        if col in df.columns:
            df = df.drop(columns=[col])

    df.to_csv(OUT_CSV, index=False)

    return df

df2 = run()
df2.head()

,review_id,department_name,state_name,year_since_first_review,star_rating,take_again,diff_index,tag_professor,num_student,post_date,...,date_missing_flag,grade_group,sentiment_label,help_ratio,difficulty_category,experience_bucket,comment_length_cat,tag_count,course_mode,professor_token
0,0,Astronomy Department,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,26,2017-06-27,...,0,B Range,Positive,0.0,Moderate,10+ yrs,Medium,3,Offline,PROF_F62E434E
1,1,Astronomy Department,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,26,2017-04-16,...,0,A Range,Positive,0.0,Easy,10+ yrs,Medium,3,Offline,PROF_F62E434E
2,2,Astronomy Department,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,26,2016-07-12,...,0,Not Reported,Positive,0.0,Moderate,10+ yrs,Medium,3,Offline,PROF_F62E434E
3,3,Astronomy Department,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,26,2014-08-12,...,0,A Range,Positive,0.0,Moderate,10+ yrs,Long,3,Offline,PROF_F62E434E
4,4,Astronomy Department,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,26,2014-02-05,...,0,Not Reported,Positive,0.0,Easy,10+ yrs,Long,3,Offline,PROF_F62E434E


In [18]:
print(df.columns)

Index(['professor_name', 'school_name', 'department_name', 'local_name',
       'state_name', 'year_since_first_review', 'star_rating', 'take_again',
       'diff_index', 'tag_professor', 'num_student', 'post_date',
       'name_onlines', 'name_not_onlines', 'student_star', 'student_difficult',
       'attence', 'for_credits', 'would_take_agains', 'grades', 'help_useful',
       'help_not_useful', 'comments', 'word_comment', 'gender', 'race',
       'asian', 'hispanic', 'nh_black', 'nh_white', 'gives_good_feedback',
       'caring', 'respected', 'participation_matters',
       'clear_grading_criteria', 'skip_class', 'amazing_lectures',
       'inspirational', 'tough_grader', 'hilarious', 'get_ready_to_read',
       'lots_of_homework', 'accessible_outside_class', 'lecture_heavy',
       'extra_credit', 'graded_by_few_things', 'group_projects', 'test_heavy',
       'so_many_papers', 'beware_of_pop_quizzes', 'IsCourseOnline',
       'post_year', 'post_month', 'date_missing_flag', 'grade_g

In [19]:
df = df.drop_duplicates(subset=["comments"])

In [22]:
import logging
from pathlib import Path
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import torch

ROOT = Path("/content")

IN_CSV = ROOT / "layer2_anonymised.csv"
EMB_PATH = ROOT / "layer3_embeddings.npy"
OUT_CSV = ROOT / "layer3_predictions.csv"

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
ZS_MODEL = "cross-encoder/nli-deberta-v3-base"

SENTIMENT_LABELS = ["positive", "negative", "neutral"]
EMOTION_LABELS = ["joy", "anger", "sadness", "fear", "surprise", "disgust"]

device = 0 if torch.cuda.is_available() else -1

def get_embeddings(texts):
    model = SentenceTransformer(SBERT_MODEL, device="cuda" if torch.cuda.is_available() else "cpu")
    return model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

def dataset_zero_shot(texts, labels):
    clf = pipeline("zero-shot-classification", model=ZS_MODEL, device=device)

    ds = Dataset.from_dict({"text": texts})

    def infer(batch):
        out = clf(batch["text"], labels)
        return {"pred": [r["labels"][0] for r in out]}

    ds = ds.map(infer, batched=True, batch_size=32)
    return ds["pred"]

def run():
    df = pd.read_csv(IN_CSV)
    log.info(f"Loaded {len(df)} rows")

    texts = df["comments"].fillna("").astype(str).tolist()

    log.info("Embedding generation...")
    emb = get_embeddings(texts)
    np.save(EMB_PATH, emb)

    log.info("Sentiment inference (dataset mode)...")
    df["sentiment"] = dataset_zero_shot(texts, SENTIMENT_LABELS)

    log.info("Emotion inference (dataset mode)...")
    df["emotion"] = dataset_zero_shot(texts, EMOTION_LABELS)

    df.to_csv(OUT_CSV, index=False)

    log.info("Layer 3 completed (GPU optimized)")

    return df

df3 = run()
df3.head()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/312 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/19957 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/19957 [00:00<?, ? examples/s]

,review_id,department_name,state_name,year_since_first_review,star_rating,take_again,diff_index,tag_professor,num_student,post_date,...,sentiment_label,help_ratio,difficulty_category,experience_bucket,comment_length_cat,tag_count,course_mode,professor_token,sentiment,emotion
0,0,Astronomy Department,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,26,2017-06-27,...,Positive,0.0,Moderate,10+ yrs,Medium,3,Offline,PROF_F62E434E,positive,joy
1,1,Astronomy Department,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,26,2017-04-16,...,Positive,0.0,Easy,10+ yrs,Medium,3,Offline,PROF_F62E434E,positive,joy
2,2,Astronomy Department,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,26,2016-07-12,...,Positive,0.0,Moderate,10+ yrs,Medium,3,Offline,PROF_F62E434E,positive,anger
3,3,Astronomy Department,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,26,2014-08-12,...,Positive,0.0,Moderate,10+ yrs,Long,3,Offline,PROF_F62E434E,positive,joy
4,4,Astronomy Department,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,26,2014-02-05,...,Positive,0.0,Easy,10+ yrs,Long,3,Offline,PROF_F62E434E,positive,joy


In [23]:
df.to_csv("layer3_enriched.csv", index=False)

In [25]:
print(df.shape)

(18922, 62)


In [26]:
embeddings = model.encode(df["comments"].fillna("").tolist(), batch_size=64)

In [27]:
import numpy as np
np.save("layer3_embeddings.npy", embeddings)

In [28]:
df.to_csv("layer3_predictions.csv", index=False)

In [30]:
df.to_csv("layer3_predictions.csv", index=False)

In [31]:
df.to_csv("layer3_predictions.csv", index=False)
np.save("layer3_embeddings.npy", embeddings)

In [33]:
df = pd.read_csv("layer3_enriched.csv")
print(df.columns)

Index(['professor_name', 'school_name', 'department_name', 'local_name',
       'state_name', 'year_since_first_review', 'star_rating', 'take_again',
       'diff_index', 'tag_professor', 'num_student', 'post_date',
       'name_onlines', 'name_not_onlines', 'student_star', 'student_difficult',
       'attence', 'for_credits', 'would_take_agains', 'grades', 'help_useful',
       'help_not_useful', 'comments', 'word_comment', 'gender', 'race',
       'asian', 'hispanic', 'nh_black', 'nh_white', 'gives_good_feedback',
       'caring', 'respected', 'participation_matters',
       'clear_grading_criteria', 'skip_class', 'amazing_lectures',
       'inspirational', 'tough_grader', 'hilarious', 'get_ready_to_read',
       'lots_of_homework', 'accessible_outside_class', 'lecture_heavy',
       'extra_credit', 'graded_by_few_things', 'group_projects', 'test_heavy',
       'so_many_papers', 'beware_of_pop_quizzes', 'IsCourseOnline',
       'post_year', 'post_month', 'date_missing_flag', 'grade_g

In [39]:
!pip install transformers accelerate

In [47]:
pip install transformers accelerate torch

In [48]:
import os
import json
import numpy as np
import pandas as pd
import faiss
import torch

from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM

# ─────────────────────────────────────
# CONFIG
# ─────────────────────────────────────
CSV_PATH = "layer3_enriched.csv"
EMB_PATH = "layer3_embeddings.npy"
INDEX_PATH = "layer4_faiss.index"
OUT_PATH = "insights.json"

TOP_K_ANN = 20
TOP_K = 8
MAX_PROFS = 100

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# ─────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────
df = pd.read_csv(CSV_PATH)
embeddings = np.load(EMB_PATH)

PROF_COL = "professor_name"

if PROF_COL not in df.columns:
    raise ValueError("Missing professor_name column in dataset")

# ─────────────────────────────────────
# FAISS INDEX (AUTO FIX)
# ─────────────────────────────────────
dim = embeddings.shape[1]

if os.path.exists(INDEX_PATH):
    print("Loading FAISS index...")
    index = faiss.read_index(INDEX_PATH)
else:
    print("Building FAISS index...")
    index = faiss.IndexHNSWFlat(dim, 32)
    index.hnsw.efConstruction = 200
    index.add(embeddings.astype("float32"))
    faiss.write_index(index, INDEX_PATH)
    print("FAISS index saved ✔")

# ─────────────────────────────────────
# RERANKER
# ─────────────────────────────────────
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# ─────────────────────────────────────
# TINYLLAMA (FREE LLM)
# ─────────────────────────────────────
print("Loading TinyLlama...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

print("TinyLlama loaded ✔")

# ─────────────────────────────────────
# HELPERS
# ─────────────────────────────────────
def get_reviews(idxs):
    return df.iloc[idxs]["comments"].fillna("").tolist()


def rerank(query, reviews):
    pairs = [(query, r) for r in reviews]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, reviews), reverse=True)
    return [r for _, r in ranked[:TOP_K]]


def generate_llm(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=180,
        temperature=0.4,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ─────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────
def run():

    professors = df[PROF_COL].unique()[:MAX_PROFS]
    results = []

    for i, prof in enumerate(professors):

        prof_df = df[df[PROF_COL] == prof]

        if len(prof_df) == 0:
            continue

        # mean embedding
        mean_vec = embeddings[prof_df.index].mean(axis=0).reshape(1, -1)

        # FAISS search
        _, idx = index.search(mean_vec.astype("float32"), TOP_K_ANN)

        reviews = get_reviews(idx[0])

        top_reviews = rerank("teaching quality", reviews)

        dept = prof_df["department_name"].mode().iloc[0] if "department_name" in prof_df else "Unknown"

        # prompt
        prompt = f"""
You are an academic assistant.

Professor: {prof}
Department: {dept}

Reviews:
{chr(10).join(top_reviews)}

Return JSON:
{{
  "summary": "",
  "strengths": [],
  "concerns": []
}}
"""

        card = generate_llm(prompt)

        results.append({
            "professor": prof,
            "department": dept,
            "card": card
        })

        if i % 20 == 0:
            print(f"Processed {i}/{len(professors)}")

    with open(OUT_PATH, "w") as f:
        json.dump(results, f, indent=2)

    print("DONE → insights.json created ✔")

# ─────────────────────────────────────
if __name__ == "__main__":
    run()

Loading FAISS index...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading TinyLlama...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

TinyLlama loaded ✔
Processed 0/100
Processed 20/100
Processed 40/100
Processed 60/100
Processed 80/100
DONE → insights.json created ✔


In [55]:
import json
import uuid
import logging
from pathlib import Path

import pandas as pd
import numpy as np

from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

ROOT = Path(".")

ENRICH_CSV = ROOT / "layer3_enriched.csv"
INSIGHTS_JSON = ROOT / "insights.json"
RLHF_PATH = ROOT / "rlhf_feedback.jsonl"

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

TOP_K = 5

df = pd.read_csv(ENRICH_CSV, low_memory=False)

# ─────────────────────────────────────────────
# LOAD INSIGHTS SAFELY
# ─────────────────────────────────────────────

INSIGHTS = {}

if INSIGHTS_JSON.exists():
    insights_list = json.loads(INSIGHTS_JSON.read_text())

    for c in insights_list:
        key = (
            c.get("professor_token")
            or c.get("professor")
            or c.get("professor_name")
        )
        if key:
            INSIGHTS[key] = c

# ─────────────────────────────────────────────
# MODEL
# ─────────────────────────────────────────────

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

chat = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True
)

# ─────────────────────────────────────────────
# CONTEXT BUILDER
# ─────────────────────────────────────────────

class ContextBuilder:
    def build(self, query: str, prof_token: str):

        prof_rows = df[df["professor_name"] == prof_token]
        reviews = prof_rows["comments"].dropna().tolist()[:TOP_K]

        raw = INSIGHTS.get(prof_token, {})
        card = raw.get("card", {})

        if isinstance(card, str):
            try:
                card = json.loads(card)
            except:
                card = {}

        if not isinstance(card, dict):
            card = {}

        return reviews, card

# ─────────────────────────────────────────────
# QA + INSIGHT ENGINE
# ─────────────────────────────────────────────

SYSTEM_PROMPT = "You are EduFeed AI. Answer only using context. Be concise."

class QAEngine:
    def __init__(self):
        self.ctx = ContextBuilder()

    # ─────────────────────────────
    # CHAT ANSWER
    # ─────────────────────────────
    def ask(self, query: str, prof_token: str):

        turn_id = str(uuid.uuid4())
        reviews, card = self.ctx.build(query, prof_token)

        context = f"""
Professor: {prof_token}
Department: {card.get('department','Unknown')}

Reviews:
{"\n---\n".join(reviews)}
"""

        prompt = f"""
{SYSTEM_PROMPT}

Context:
{context}

Question:
{query}

Answer:
"""

        output = chat(prompt)[0]["generated_text"]
        answer = output.split("Answer:")[-1].strip()

        print(answer)
        return turn_id, answer

    # ─────────────────────────────
    # ⭐ NEW: PROFESSOR INSIGHT REPORT
    # ─────────────────────────────
    def generate_professor_report(self, prof_token: str):

        reviews, card = self.ctx.build("", prof_token)

        text = "\n".join(reviews[:20])

        prompt = f"""
You are an academic feedback analyst.

Create a PROFESSOR INSIGHT REPORT.

Include:
1. Overall sentiment (Positive / Negative / Neutral)
2. Top 3 positive reasons
3. Top 3 negative reasons
4. Key student evidence (quotes)
5. Final actionable advice for improvement

Rules:
- Be honest and factual
- Use ONLY given feedback
- Keep it structured

Professor: {prof_token}

Feedback:
{text}

OUTPUT:
"""

        result = chat(prompt)[0]["generated_text"]
        return result

# ─────────────────────────────────────────────
# RLHF STORE
# ─────────────────────────────────────────────

class RLHFStore:
    def submit(self, turn_id, prof_token, query, answer, rating):

        record = {
            "id": str(uuid.uuid4()),
            "turn_id": turn_id,
            "professor": prof_token,
            "query": query,
            "answer": answer[:200],
            "rating": rating,
        }

        with open(RLHF_PATH, "a") as f:
            f.write(json.dumps(record) + "\n")

        return record

# ─────────────────────────────────────────────
# MAIN API
# ─────────────────────────────────────────────

class EduFeedChat:
    def __init__(self):
        self.engine = QAEngine()
        self.feedback = RLHFStore()

    def ask(self, query, prof_token):
        return self.engine.ask(query, prof_token)

    def get_professor_insight(self, prof_token):
        return self.engine.generate_professor_report(prof_token)

    def submit_feedback(self, turn_id, prof_token, query, answer, rating):
        return self.feedback.submit(turn_id, prof_token, query, answer, rating)

# ─────────────────────────────────────────────
# TEST
# ─────────────────────────────────────────────

if __name__ == "__main__":

    app = EduFeedChat()

    prof = df["professor_name"].iloc[0]

    print("\n🔵 CHAT TEST")
    q = "Is this professor strict?"
    turn_id, ans = app.ask(q, prof)

    print("\n🟢 PROFESSOR REPORT")
    report = app.get_professor_insight(prof)
    print(report)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔵 CHAT TEST


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


No. Professor Looney is an extremely laid-back professor who usually lets everyone work at their own pace. However, he does have some expectations which include completing class work and weekly assignments. If you feel like you may not be able to meet his expectations, it's best to ask him for extra help or to do the assignments on your own.

🟢 PROFESSOR REPORT

You are an academic feedback analyst.

Create a PROFESSOR INSIGHT REPORT.

Include:
1. Overall sentiment (Positive / Negative / Neutral)
2. Top 3 positive reasons
3. Top 3 negative reasons
4. Key student evidence (quotes)
5. Final actionable advice for improvement

Rules:
- Be honest and factual
- Use ONLY given feedback
- Keep it structured

Professor: Leslie  Looney

Feedback:
This class is hard, but its a two-in-one gen-ed knockout, and the content is very stimulating. Unlike most classes, you have to actually participate to pass. Sections are easy and offer extra credit every week. Very funny dude. Not much more I can say.


In [56]:

import json
import shutil
import logging
from pathlib import Path
from datetime import datetime

import pandas as pd

# ── ROOT (FIXED FOR COLAB + LOCAL) ─────────────────────────
ROOT = Path.cwd()   # <-- FIX instead of __file__
DATA = ROOT

OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

LOG_PATH = ROOT / "layer6.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)


# ── FILE PATHS ─────────────────────────────────────────────
ENRICH_CSV = DATA / "layer3_enriched.csv"
INSIGHTS_JSON = DATA / "insights.json"


# ── 6.1 EXPORT PREDICTIONS ────────────────────────────────
def export_predictions():
    if not ENRICH_CSV.exists():
        log.warning("layer3_enriched.csv not found")
        return

    df = pd.read_csv(ENRICH_CSV, low_memory=False)
    out = OUTPUTS / "predictions.csv"
    df.to_csv(out, index=False)

    log.info(f"predictions.csv saved → {len(df)} rows")


# ── 6.2 EXPORT INSIGHTS ───────────────────────────────────
def export_insights():
    if INSIGHTS_JSON.exists():
        shutil.copy2(INSIGHTS_JSON, OUTPUTS / "insights.json")
        log.info("insights.json copied")
    else:
        log.warning("insights.json missing")


# ── 6.3 REPORT GENERATION (SAFE) ──────────────────────────
def export_report():
    out = OUTPUTS / "report.md"

    df = pd.read_csv(ENRICH_CSV, low_memory=False) if ENRICH_CSV.exists() else None

    sentiment = {}
    if df is not None and "nlp_sentiment" in df.columns:
        sentiment = df["nlp_sentiment"].value_counts().to_dict()

    rating_avg = df["star_rating"].mean() if df is not None and "star_rating" in df.columns else "N/A"

    text = f"""
# EduFeed Report

Generated: {datetime.now()}

## Dataset
Rows: {len(df) if df is not None else 0}

## Sentiment
{sentiment}

## Avg Rating
{rating_avg}

## Status
Pipeline completed successfully.
"""

    out.write_text(text)
    log.info("report.md created")


# ── RUN ALL EXPORTS ───────────────────────────────────────
def run_exports():
    log.info("Running Layer 6 exports...")
    export_predictions()
    export_insights()
    export_report()
    log.info("Layer 6 DONE ✔")


# ── ENTRY ────────────────────────────────────────────────
if __name__ == "__main__":
    run_exports()

In [63]:
import sys
from pathlib import Path

ROOT = Path.cwd()   # IMPORTANT (NOT __file__)
sys.path.append(str(ROOT))

In [64]:
import os
print(os.listdir("."))

['.config', 'layer1_report.json', 'layer3_enriched.csv', 'layer3_predictions.csv', 'insights.json', 'outputs', 'layer2_prof_map.json', 'layer4_faiss.index', 'layer1_ingested.csv', 'layer2.log', 'layer3_embeddings.npy', 'rlhf_feedback.jsonl', 'layer2_anonymised.csv', 'edufeed_clean_data.xlsx', 'layer2_anonymized.csv', 'layer1.log', 'sample_data']


In [74]:
import argparse
from pathlib import Path

ROOT = Path.cwd()

EXPECTED_ARTIFACTS = {
    1: ["layer1_ingested.csv", "layer1_report.json"],
    2: ["layer2_anonymised.csv", "layer2_prof_map.json"],
    3: ["layer3_enriched.csv", "layer3_embeddings.npy"],
    4: ["layer4_faiss.index", "insights.json"],
    5: ["rlhf_feedback.jsonl"],
    6: ["outputs"]
}


def check_layer(layer: int):
    items = EXPECTED_ARTIFACTS.get(layer, [])
    missing = []
    present = []

    for item in items:
        if (ROOT / item).exists():
            present.append(item)
        else:
            missing.append(item)

    print("\n" + "=" * 60)
    print(f"LAYER {layer} STATUS")
    print("=" * 60)

    if present:
        print("\nPresent:")
        for p in present:
            print(f"  OK  {p}")

    if missing:
        print("\nMissing:")
        for m in missing:
            print(f"  X   {m}")
    else:
        print("\nAll required artifacts are present.")


def run_pipeline(layers):
    print("\nPIPELINE STATUS REPORT\n")

    for layer in layers:
        if layer == 5:
            print("\nLAYER 5: Runtime chat service (run separately)")
            continue

        check_layer(layer)

    print("\nPIPELINE CHECK COMPLETE")


# ---- RUN HERE ----
run_pipeline([1,2,3,4,5,6])


PIPELINE STATUS REPORT


LAYER 1 STATUS

Present:
  OK  layer1_ingested.csv
  OK  layer1_report.json

All required artifacts are present.

LAYER 2 STATUS

Present:
  OK  layer2_anonymised.csv
  OK  layer2_prof_map.json

All required artifacts are present.

LAYER 3 STATUS

Present:
  OK  layer3_enriched.csv
  OK  layer3_embeddings.npy

All required artifacts are present.

LAYER 4 STATUS

Present:
  OK  layer4_faiss.index
  OK  insights.json

All required artifacts are present.

LAYER 5: Runtime chat service (run separately)

LAYER 6 STATUS

Present:
  OK  outputs

All required artifacts are present.

PIPELINE CHECK COMPLETE


In [82]:
from transformers import pipeline
import json
from datetime import datetime

# ─────────────────────────────
# MODELS
# ─────────────────────────────
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

llm = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)


# ─────────────────────────────
# ANALYZE SINGLE FEEDBACK ONLY
# ─────────────────────────────
def analyze_feedback():

    print("\n=== EduFeed Feedback Form ===\n")

    name = input("Enter your name: ")
    email = input("Enter email: ")
    roll = input("Enter roll number: ")

    prof = input("Enter professor name: ")
    rating = int(input("Enter rating (1-5): "))
    feedback = input("Enter feedback: ")

    # anonymized record
    record = {
        "anon_id": str(hash(name + email + roll)),
        "professor": prof,
        "rating": rating,
        "feedback": feedback,
        "timestamp": str(datetime.utcnow())
    }

    print("\n✔ Feedback stored anonymously")
    print(record)

    # ─────────────────────────────
    # SENTIMENT (ONLY THIS INPUT)
    # ─────────────────────────────
    sentiment = sentiment_model(feedback[:512])[0]["label"]

    print("\n=== SENTIMENT RESULT ===")
    print("Sentiment:", sentiment)

    # ─────────────────────────────
    # LLM INSIGHT (ONLY THIS INPUT)
    # ─────────────────────────────
    prompt = f"""
You are an academic feedback analyst.

Analyze ONLY this single feedback:

Professor: {prof}
Rating: {rating}
Feedback: {feedback}

Return:
1. Sentiment interpretation
2. Why this sentiment occurred
3. Professor improvement suggestion
4. Student perception summary

Keep response under 300-500 words.
Do NOT add extra examples.
"""

    output = llm(
        prompt,
        max_new_tokens=250,
        do_sample=False
    )[0]["generated_text"]

    # cleanup prompt leakage
    if "Return:" in output:
        output = output.split("Return:")[-1]

    print("\n=== AI PROFESSOR ANALYSIS ===\n")
    print(output)


# ─────────────────────────────
# RUN
# ─────────────────────────────
if __name__ == "__main__":
    analyze_feedback()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


=== EduFeed Feedback Form ===

Enter your name: Rajesh
Enter email: rajesh@gmail.com
Enter roll number: 812624
Enter professor name: prof john
Enter rating (1-5): 4
Enter feedback: he is good


/tmp/ipykernel_835/609454752.py:40: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": str(datetime.utcnow())
Both `max_new_tokens` (=250) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✔ Feedback stored anonymously
{'anon_id': '7023657891939623019', 'professor': 'prof john', 'rating': 4, 'feedback': 'he is good', 'timestamp': '2026-04-21 20:43:02.099049'}

=== SENTIMENT RESULT ===
Sentiment: POSITIVE

=== AI PROFESSOR ANALYSIS ===


1. Sentiment interpretation
2. Why this sentiment occurred
3. Professor improvement suggestion
4. Student perception summary

Keep response under 300-500 words.
Do NOT add extra examples.

[Insert logo]

[Insert company name]

[Insert company website]

[Insert company email]

[Insert company phone number]

[Insert company address]

[Insert company logo]

[Insert company name]

[Insert company website]

[Insert company email]

[Insert company phone number]

[Insert company address]

[Insert company logo]

[Insert company name]

[Insert company website]

[Insert company email]

[Insert company phone number]

[Insert company address]

[Insert company logo]

[Insert company name]

[Insert company website]

[Insert company email]

[Insert com

In [83]:
from transformers import pipeline
from datetime import datetime
import json
import re

# ─────────────────────────────
# MODELS
# ─────────────────────────────
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

llm = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

# ─────────────────────────────
# SENTIMENT
# ─────────────────────────────
def get_sentiment(text):
    return sentiment_model(text[:512])[0]["label"]


# ─────────────────────────────
# CLEAN LLM OUTPUT
# ─────────────────────────────
def clean_output(text: str) -> str:
    bad_patterns = [
        "insert company", "logo", "website",
        "phone number", "address", "[Insert"
    ]
    for b in bad_patterns:
        text = text.replace(b, "")

    # remove repeated blank noise
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# ─────────────────────────────
# MAIN ANALYZER
# ─────────────────────────────
def analyze_feedback():

    print("\n=== EduFeed Feedback Form ===\n")

    name = input("Enter name: ")
    email = input("Enter email: ")
    roll = input("Enter roll no: ")

    professor = input("Enter professor name: ")
    rating = int(input("Enter rating (1-5): "))
    feedback = input("Enter feedback: ")

    record = {
        "anon_id": hash(name + email + roll),
        "professor": professor,
        "rating": rating,
        "feedback": feedback,
        "timestamp": str(datetime.now())
    }

    print("\n✔ Feedback stored anonymously")
    print(record)

    # ───── SENTIMENT ─────
    sentiment = get_sentiment(feedback)

    print("\n=== SENTIMENT RESULT ===")
    print(sentiment)

    # ───── LLM PROMPT (STRICT) ─────
    prompt = f"""
You are a strict academic feedback analyzer.

IMPORTANT RULES:
- Do NOT generate templates
- Do NOT add fake information
- Do NOT include logos, websites, or contact details
- Keep output under 300 words
- Only analyze given feedback

INPUT:
Professor: {professor}
Rating: {rating}
Feedback: {feedback}

OUTPUT FORMAT:

Sentiment Interpretation:
Reason:
Key Strength:
Key Weakness:
Improvement Suggestion:
Student Summary:
"""

    output = llm(
        prompt,
        max_new_tokens=200,
        do_sample=False,
        temperature=0.2,
        repetition_penalty=1.3,
        return_full_text=False
    )[0]["generated_text"]

    output = clean_output(output)

    print("\n=== AI PROFESSOR ANALYSIS ===\n")
    print(output)


# ─────────────────────────────
# RUN
# ─────────────────────────────
if __name__ == "__main__":
    analyze_feedback()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


=== EduFeed Feedback Form ===

Enter name: Rajesh
Enter email: rajesh@gmail.com
Enter roll no: 8276
Enter professor name: john
Enter rating (1-5): 4
Enter feedback: he is good


Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✔ Feedback stored anonymously
{'anon_id': 4962326805341421757, 'professor': 'john', 'rating': 4, 'feedback': 'he is good', 'timestamp': '2026-04-21 20:45:11.222773'}

=== SENTIMENT RESULT ===
POSITIVE

=== AI PROFESSOR ANALYSIS ===

Overall Analysis:
Conclusion:

PROFESSOR JOHN'S REVIEW OF THE STUDENT’s PAPER (Given Feedback)
The paper you submitted was well written and presented the topic in an organized manner with proper citations of sources used throughout. However, I noticed that there were some grammatical errors such as incorrect use of commas and missing punctuation marks at times. Additionally, your writing style could benefit from more clarity on certain points to ensure consistency across all sections of the essay. Overall, while it may not be perfect, this student has demonstrated their ability to write effectively for academia. Based on these observations, here are my recommendations for improvement:

1. Grammar/Spelling Checking - Please make sure to double check each se

In [84]:
import shutil
from pathlib import Path

ROOT = Path(".")

SAVE_DIR = Path("edufeed_saved_model")
SAVE_DIR.mkdir(exist_ok=True)

files_to_save = [
    "layer1_ingested.csv",
    "layer1_report.json",
    "layer2_anonymised.csv",
    "layer2_prof_map.json",
    "layer3_enriched.csv",
    "layer3_embeddings.npy",
    "layer3_predictions.csv",
    "layer4_faiss.index",
    "insights.json",
    "rlhf_feedback.jsonl"
]

for f in files_to_save:
    p = ROOT / f
    if p.exists():
        shutil.copy(p, SAVE_DIR / f)

print("MODEL SNAPSHOT SAVED ✔")
print("Location:", SAVE_DIR)

MODEL SNAPSHOT SAVED ✔
Location: edufeed_saved_model


In [85]:
import shutil

shutil.make_archive("edufeed_model", "zip", "edufeed_saved_model")

'/content/edufeed_model.zip'

In [86]:
import json
import numpy as np
import pandas as pd
import faiss
from pathlib import Path

MODEL_DIR = Path("edufeed_saved_model")

df = pd.read_csv(MODEL_DIR / "layer3_enriched.csv")
embeddings = np.load(MODEL_DIR / "layer3_embeddings.npy")

index = faiss.read_index(str(MODEL_DIR / "layer4_faiss.index"))

insights = json.loads((MODEL_DIR / "insights.json").read_text())

print("MODEL LOADED ✔")

MODEL LOADED ✔


In [88]:
import os

files = [
    "layer3_enriched.csv",
    "layer3_embeddings.npy",
    "layer4_faiss.index",
    "insights.json",
    "layer2_prof_map.json",
    "rlhf_feedback.jsonl"
]

print("=== MODEL SAVE CHECK ===\n")

for f in files:
    if os.path.exists(f):
        print("OK  ", f)
    else:
        print("MISSING ", f)

=== MODEL SAVE CHECK ===

OK   layer3_enriched.csv
OK   layer3_embeddings.npy
OK   layer4_faiss.index
OK   insights.json
OK   layer2_prof_map.json
OK   rlhf_feedback.jsonl


In [89]:
import pandas as pd

df = pd.read_csv("layer3_enriched.csv")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

Rows: 18922
Columns: ['professor_name', 'school_name', 'department_name', 'local_name', 'state_name', 'year_since_first_review', 'star_rating', 'take_again', 'diff_index', 'tag_professor', 'num_student', 'post_date', 'name_onlines', 'name_not_onlines', 'student_star', 'student_difficult', 'attence', 'for_credits', 'would_take_agains', 'grades', 'help_useful', 'help_not_useful', 'comments', 'word_comment', 'gender', 'race', 'asian', 'hispanic', 'nh_black', 'nh_white', 'gives_good_feedback', 'caring', 'respected', 'participation_matters', 'clear_grading_criteria', 'skip_class', 'amazing_lectures', 'inspirational', 'tough_grader', 'hilarious', 'get_ready_to_read', 'lots_of_homework', 'accessible_outside_class', 'lecture_heavy', 'extra_credit', 'graded_by_few_things', 'group_projects', 'test_heavy', 'so_many_papers', 'beware_of_pop_quizzes', 'IsCourseOnline', 'post_year', 'post_month', 'date_missing_flag', 'grade_group', 'sentiment_label', 'help_ratio', 'difficulty_category', 'experience_b

In [90]:
import faiss

index = faiss.read_index("layer4_faiss.index")
print("FAISS vectors:", index.ntotal)

FAISS vectors: 18922


In [92]:
import json

with open("insights.json", "r") as f:
    data = json.load(f)

print("Number of professor insights:", len(data))

print("\nSample entries:\n")

# show first 3 safely
for item in data[:3]:
    print(item)

Number of professor insights: 100

Sample entries:

{'professor': 'Leslie  Looney', 'department': 'Astronomy Department', 'card': '\nYou are an academic assistant.\n\nProfessor: Leslie  Looney\nDepartment: Astronomy Department\n\nReviews:\nGood professor. Not the best teacher, but he keeps the class is plenty entertaining and fairly easy.\nI loved this class. One of my favorites so far. He is a great teacher and person. Assignments aren\\\'t bad and there aren\\\'t tests. He is very nice and easy to get along with.\nThis professor is great. this class only involves teaching the class for a day and writing two papers. Very easy and fun, just watching him teach makes you smile. You will have a blast\nAmazing professor! One of the few professors that has left an everylasting impression. I learned so much in his class. He makes everything interesting! However, the class does require a lot of assignments but you\\\'ll do them and not care. He is just that awesome. You have labs where you ne

In [93]:
import shutil
from pathlib import Path

files = [
    "layer3_enriched.csv",
    "layer3_embeddings.npy",
    "layer4_faiss.index",
    "insights.json",
    "layer2_prof_map.json",
    "rlhf_feedback.jsonl",
    "layer3_predictions.csv",
    "layer1_report.json"
]

package_dir = Path("edufeed_model_package")
package_dir.mkdir(exist_ok=True)

for f in files:
    if Path(f).exists():
        shutil.copy(f, package_dir / f)

print("Package created at:", package_dir)

Package created at: edufeed_model_package


In [94]:
import shutil

zip_name = "edufeed_model"

shutil.make_archive(zip_name, 'zip', "edufeed_model_package")

print("ZIP created:", zip_name + ".zip")

ZIP created: edufeed_model.zip


In [95]:
from google.colab import files

files.download("edufeed_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>